In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install transformers datasets scikit-learn


In [ ]:
# Step 2: Load the CSV from your Drive path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

csv_path = 'specify'
df = pd.read_csv(csv_path)

# Step 3: Show unique class_id values
print("Unique class_id labels:")
print(df['class_id'].unique())


Unique class_id labels:
['a82_date_time' 'a90_email' 'location' 'a111_name_all' 'a49_phone' 'safe']


In [ ]:
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder

tokenizer = AutoTokenizer.from_pretrained("microsoft/mpnet-base")
# Assume df is your full dataset
df_train = df[df['split'].isin(['train', 'val'])].copy()
df_test = df[df['split'] == 'test'].copy()

# Encode class labels
le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['class_id'])
df_test['label'] = le.transform(df_test['class_id'])

# Hugging Face Dataset
ds_train = Dataset.from_pandas(df_train[['text', 'label']])
ds_test = Dataset.from_pandas(df_test[['text', 'label']])
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

ds_train = ds_train.map(tokenize, batched=True)
ds_test = ds_test.map(tokenize, batched=True)


Map:   0%|          | 0/12315 [00:00<?, ? examples/s]

Map:   0%|          | 0/4917 [00:00<?, ? examples/s]

In [ ]:
!pip install --upgrade transformers


In [ ]:
import transformers
print(transformers.__version__)


4.53.1


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
import numpy as np
import matplotlib.pyplot as plt
import os

# Load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained("microsoft/mpnet-base", num_labels=len(le.classes_))
tokenizer = AutoTokenizer.from_pretrained("microsoft/mpnet-base")

# Training arguments
args = TrainingArguments(
    output_dir="./mpnet_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# Metric computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

# Train and save model
trainer.train()
trainer.save_model("./mpnet_output/final_model")
tokenizer.save_pretrained("./mpnet_output/final_model")


Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at microsoft/mpnet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-8-3209551356.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.885500,0.686756,0.756356,0.752409
2,0.495600,0.676367,0.768965,0.767148
3,0.419000,0.672173,0.772829,0.772284


('./mpnet_output/final_model/tokenizer_config.json',
 './mpnet_output/final_model/special_tokens_map.json',
 './mpnet_output/final_model/vocab.txt',
 './mpnet_output/final_model/added_tokens.json',
 './mpnet_output/final_model/tokenizer.json')

In [ ]:
import torch

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(ds_test)
print("Test Results:", test_results)

# Predict + classification report
preds = trainer.predict(ds_test)
y_pred = torch.argmax(torch.tensor(preds.predictions), dim=-1)
y_true = preds.label_ids
print(classification_report(le.inverse_transform(y_true), le.inverse_transform(y_pred.numpy())))

Test Results: {'eval_loss': 0.6721725463867188, 'eval_accuracy': 0.7728289607484239, 'eval_f1': 0.772283722321406, 'eval_runtime': 175.7377, 'eval_samples_per_second': 27.979, 'eval_steps_per_second': 0.876, 'epoch': 3.0}
               precision    recall  f1-score   support

a111_name_all       0.72      0.80      0.76      1081
    a49_phone       0.78      0.79      0.79        82
a82_date_time       0.91      0.90      0.90      1440
    a90_email       0.97      0.97      0.97       253
     location       0.82      0.56      0.67      1353
         safe       0.56      0.80      0.66       708

     accuracy                           0.77      4917
    macro avg       0.79      0.80      0.79      4917
 weighted avg       0.79      0.77      0.77      4917

